<a href="https://colab.research.google.com/github/jlloring/ST-554_JLoring/blob/main/Loring_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ST 554 Homework 7**
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

In [68]:
# importing required modules
import pandas as pd
import numpy as np
from sklearn import linear_model
from math import sqrt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, Ridge, ElasticNet, \
                                 LogisticRegressionCV, LassoCV, RidgeCV, ElasticNetCV
from sklearn.preprocessing import PolynomialFeatures

## **Data Work**
The code below reads in the `winequality-red.csv` and `winequality-white.csv` files from the [UCI machine learning repository site](https://archive.ics.uci.edu/dataset/186/wine+quality). These files must be re-uploaded to the *Files* section at the start of each new session!

In [2]:
white = pd.read_csv("winequality-white.csv", sep = ';')
red = pd.read_csv("winequality-red.csv", sep = ';')

Now, we create a variable called `type` within each dataset that represents the type of wine (i.e., red or white). Since we know this binary identification will be used for logistic regression, we will also create a variable called `type_num` that takes on a value of 1 for red wine and 0 for white wine.

In [3]:
white['type'] = 'White'
white['type_num'] = 0
red['type'] = 'Red'
red['type_num'] = 1

To verify that our data is looking as expected, we use the `.head()` method on each dataset.

In [4]:
# check white wine data
white.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_num
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6,White,0
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6,White,0
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6,White,0
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,White,0
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,White,0


In [5]:
#check red wine data
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_num
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,Red,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,Red,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,Red,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,Red,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,Red,1


Now, we use the `concat()` function from `pandas` to combine these two datasets.

In [6]:
wine = pd.concat([white, red])
wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_num
0,7.0,0.270,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6,White,0
1,6.3,0.300,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6,White,0
2,8.1,0.280,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6,White,0
3,7.2,0.230,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6,White,0
4,7.2,0.230,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6,White,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5,Red,1
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6,Red,1
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6,Red,1
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5,Red,1


We now have our final working dataset, `wine`. According to the UCI machine learning repository, there are no missing values, so we do not need to do any additional data cleaning!

## **Splitting the Data**
The code below splits up the `wine` data into a training and test set. Stratified sampling is used to make sure that we have a similar proportion of white and red wines in the training and test sets. The `train_test_split()` function from `sklearn.model_selection` will be used. This splitting is done two separate times for our two separate tasks:
1. **Regression Task** (`alcohol` as Response)
2. **Classification Task** (`type_num` as Response)

#### **Regression Task Splitting**
The code below splits the `wine` dataset into training and testing data for use with the Regression Task.

In [7]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
wine.drop(['alcohol', 'type'], axis = 1), #drops the type category text, keeps binary version
wine['alcohol'],
stratify = wine['type'], #stratifies on wine type
test_size = 0.20,
random_state = 44)

##### **Regression Task Standardizing**
Now that we have our training and testing data, we want to standardize our values! The code below accomplishes this.

In [8]:
means_reg = X_train1.apply(np.mean, axis = 0) #grabs means of all training data columns
stds_reg = X_train1.apply(np.std, axis = 0) #grabs standard deviations of all training data columns

#overwrites training data with standardized values
X_train1 = X_train1.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)
X_train1.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type_num
3909,-0.782920,0.302956,-0.542342,-0.145335,-0.535424,0.819461,0.388449,-1.019322,0.072989,-0.681980,1.366087,-0.571351
2624,-0.821662,-0.792999,0.349127,-0.482879,-0.981264,-0.603268,-0.642021,-1.353087,1.065656,-0.413948,-0.940086,-0.571351
906,0.844258,-0.427681,0.897723,-0.883712,-0.228909,-1.286178,-0.784155,-0.113391,-0.857636,0.524163,-2.093173,-0.571351
1228,1.696589,-0.488567,1.446320,-0.904808,-0.256774,-0.318722,-0.126786,-0.896715,0.072989,-1.687099,0.213000,-0.571351
1011,0.379350,-1.097431,0.623425,-0.799326,-0.228909,1.673099,1.010285,-0.522082,0.135030,0.859202,1.366087,-0.571351


*Note: The standardizations applied to the training data above will be used on the test data as well!*

#### **Classification Task Splitting**
The code below splits the `wine` dataset into training and testing data for use with the Classification Task.

In [9]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
wine.drop(['type_num', 'type'], axis = 1), #drops the type category text, keeps binary version
wine['type_num'],
stratify = wine['type'], #stratifies on wine type
test_size = 0.20,
random_state = 44)

##### **Classification Task Standardizing**
Now that we have our training and testing data, we want to standardize our values! The code below accomplishes this.

In [10]:
means_cls = X_train2.apply(np.mean, axis = 0) #grabs means of all training data columns
stds_cls = X_train2.apply(np.std, axis = 0) #grabs standard deviations of all training data columns

#overwrites training data with standardized values
X_train2 = X_train2.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)
X_train2.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
3909,-0.782920,0.302956,-0.542342,-0.145335,-0.535424,0.819461,0.388449,-1.019322,0.072989,-0.681980,0.620562,1.366087
2624,-0.821662,-0.792999,0.349127,-0.482879,-0.981264,-0.603268,-0.642021,-1.353087,1.065656,-0.413948,1.298066,-0.940086
906,0.844258,-0.427681,0.897723,-0.883712,-0.228909,-1.286178,-0.784155,-0.113391,-0.857636,0.524163,-0.141630,-2.093173
1228,1.696589,-0.488567,1.446320,-0.904808,-0.256774,-0.318722,-0.126786,-0.896715,0.072989,-1.687099,1.721506,0.213000
1011,0.379350,-1.097431,0.623425,-0.799326,-0.228909,1.673099,1.010285,-0.522082,0.135030,0.859202,0.620562,1.366087


*Note: The standardizations applied to the training data above will be used on the test data as well!*

Now that our data is split for each task, we can start making our models!

## **Regression Task: `alcohol` as Response**
In this section, we use a variety of regression techniques to fit both training models and tests models. We will use:
- Multiple Linear Regression (MLR)
- LASSO
- Ridge Regression
- Elastic Net

### **Train Models**
In this sub-section, we fit the four types of models described above on our training data.

#### **Multiple Linear Regression**
Here are the four different MLR models that I will choose to fit (listed predictors):
1. `citric acid`, `residual sugar`, `pH`
2. `volatile acidity`, `residual sugar`, `total sulfur dioxide`, `density`, `type_num`
3. `fixed acidity`, `pH`, and their interaction
4. `residual sugar`, `density`, `residual sugar`$^{2}$, `density`$^{2}$, the interaction of density and residual sugar

##### **MLR #1**
The code below fits the first model described at the beginning of the MLR section.

In [11]:
MLR_1_cols = ['citric acid', 'residual sugar', 'pH']
MLR_1 = linear_model.LinearRegression().fit(X_train1[MLR_1_cols],
                                            y_train1)
print(MLR_1.intercept_, MLR_1.coef_)

10.467236867423512 [ 0.06027543 -0.43050488  0.03963814]


The order of the coefficients corresponds to the order of the columns in the `wine` dataframe. Therefore, the correct order is `citric acid`, `residual sugar`, and `pH`, respectively.

##### **MLR #2**
The code below fits the second model described at the beginning of the MLR section.

In [12]:
MLR_2_cols = ['volatile acidity', 'residual sugar', 'total sulfur dioxide',
              'density', 'type_num']
MLR_2 = linear_model.LinearRegression().fit(X_train1[MLR_2_cols],
                                            y_train1)
print(MLR_2.intercept_, MLR_2.coef_)

10.467236867423505 [-0.03002403  0.66125381 -0.11014287 -1.45206785  0.72135357]


The order of the coefficients corresponds to the order of the columns in the `wine` dataframe. Therefore, the correct order is `volatile acidity`, `residual sugar`, `total sulfur dioxide`, `density`, and `type_num`, respectively.

##### **MLR #3**
The code below fits the third model described at the beginning of the MLR section. This requires multiple steps since we are using an interaction term in this model.

In [13]:
MLR_3_cols = ['fixed acidity', 'pH']

#creates interaction object
poly3 = PolynomialFeatures(interaction_only=True, include_bias = False)

#fits interaction object with variables we want
design3 = poly3.fit_transform(X_train1[MLR_3_cols])

#fits MLR model with interactions
MLR_3 = linear_model.LinearRegression().fit(X = design3, y = y_train1)

print(MLR_3.intercept_, MLR_3.coef_)

10.433094946500994 [-0.10007542  0.09546757 -0.13761727]


We can use the `.get_feature_names_out()` method to confirm the order of the coefficients since we are using `PolynomialFeatures`.

In [14]:
poly3.get_feature_names_out(['fixed acidity', 'pH'])

array(['fixed acidity', 'pH', 'fixed acidity pH'], dtype=object)

Therefore, the order of the coefficients is `fixed acidity`, `pH`, and their interaction, respectively.

##### **MLR #4**
The code below fits the fourth and final model described at the beginning of the MLR section. This also requires multiple steps since we are using both quadratic terms and a interaction term in this model.

In [15]:
MLR_4_cols = ['residual sugar', 'density']

#creates interaction & polynomial object
poly4 = PolynomialFeatures(degree = 2, interaction_only = False, include_bias = False)

#fits interaction object with variables we want
design4 = poly4.fit_transform(X_train1[MLR_4_cols])

#fits MLR model with interactions
MLR_4 = linear_model.LinearRegression().fit(X = design4, y = y_train1)

print(MLR_4.intercept_, MLR_4.coef_)

10.112001788632426 [ 0.22766094 -1.01870269  0.26248177 -0.68736313  0.46672017]


Again, we can use the `.get_feature_names_out()` method to confirm the order of the coefficients since we are using `PolynomialFeatures`.

In [16]:
poly4.get_feature_names_out(['residual sugar', 'density'])

array(['residual sugar', 'density', 'residual sugar^2',
       'residual sugar density', 'density^2'], dtype=object)

Therefore, the order of the coefficients is `residual sugar`, `density`, `residual sugar`$^{2}$, the interaction between residual sugar and density, and `density`$^{2}$, respectively.

##### **CV with MLR Models**
The code below uses cross-validation with each of the above MLR models to help select the best one.

In [17]:
#cross-validation on MLR #1
cv1_MLR = cross_validate(linear_model.LinearRegression(),
                         X_train1[MLR_1_cols],
                         y_train1,
                         cv = 5,
                         scoring = 'neg_mean_squared_error',
                         return_train_score = True)

#cross-validation on MLR #2
cv2_MLR = cross_validate(linear_model.LinearRegression(),
                         X_train1[MLR_2_cols],
                         y_train1,
                         cv = 5,
                         scoring = 'neg_mean_squared_error',
                         return_train_score = True)

#cross-validation on MLR #3
cv3_MLR = cross_validate(linear_model.LinearRegression(),
                         design3,
                         y_train1,
                         cv = 5,
                         scoring = 'neg_mean_squared_error',
                         return_train_score = True)

#cross-validation on MLR #4
cv4_MLR = cross_validate(linear_model.LinearRegression(),
                         design4,
                         y_train1,
                         cv = 5,
                         scoring = 'neg_mean_squared_error',
                         return_train_score = True)

Now that cross-validation is complete, we can compare the CV RMSE values to select the best model. Lower is better!

In [18]:
print(
    round(np.sqrt(-sum(cv1_MLR["test_score"])/5),4),
    round(np.sqrt(-sum(cv2_MLR["test_score"])/5),4),
    round(np.sqrt(-sum(cv3_MLR["test_score"])/5),4),
    round(np.sqrt(-sum(cv4_MLR["test_score"])/5),4)
    )

1.098 0.6905 1.1638 0.7272


Since the RMSE is lowest for the second MLR model, that is the best choice of the four fitted models!

In [19]:
MLR_best_cols = MLR_2_cols
MLR_best = MLR_2
print(MLR_best.intercept_)
print(np.array(list(zip(X_train1[MLR_best_cols], MLR_best.coef_))))

10.467236867423505
[['volatile acidity' '-0.03002402751932421']
 ['residual sugar' '0.6612538082282409']
 ['total sulfur dioxide' '-0.11014286805865692']
 ['density' '-1.4520678529538855']
 ['type_num' '0.7213535715000869']]


#### **LASSO**
The code below fits a LASSO model with five predictors:
- fixed acidity
- citric acid
- density
- pH
- quality

Since we want to use CV to select the tuning parameter, we will use the `LassoCV` function.

In [20]:
lasso_reg_cols = ['fixed acidity', 'citric acid', 'density', 'pH', 'quality']
lasso_reg = LassoCV(cv = 5, random_state = 44) \
            .fit(X_train1[lasso_reg_cols],
                 y_train1)

Now, we see which $\alpha$ corresponds to the lowest MSE value!

In [21]:
np.set_printoptions(suppress = True) #code provided in notes
fit_info_lasso_reg = np.array(list(zip(lasso_reg.alphas_, np.mean(lasso_reg.mse_path_, axis = 1))))
fit_info_lasso_reg[fit_info_lasso_reg[:,1].argsort()][0:10,].round(7)

array([[0.0017659, 0.5034068],
       [0.0016469, 0.5034069],
       [0.0018935, 0.503407 ],
       [0.0015359, 0.5034072],
       [0.0020304, 0.5034074],
       [0.0014324, 0.5034075],
       [0.0013358, 0.503408 ],
       [0.0021771, 0.5034081],
       [0.0012458, 0.5034087],
       [0.0023344, 0.5034092]])

Therefore, the tuning parameter that should be selected for this LASSO model is $\alpha = 0.0017659$. We now store the fit of this LASSO model with the best tuning parameter for later use with the test data.

In [22]:
lasso_reg_best = Lasso(lasso_reg.alpha_, random_state = 44) \
                 .fit(X_train1[lasso_reg_cols], y_train1)

print(lasso_reg_best.intercept_)
print(np.array(list(zip(X_train1[lasso_reg_cols], lasso_reg_best.coef_))))

10.467236867423507
[['fixed acidity' '0.408716223672921']
 ['citric acid' '0.0018428615253397237']
 ['density' '-0.9350933681421721']
 ['pH' '0.24723879142951843']
 ['quality' '0.25379510519136994']]


#### **Ridge Regression**
The code below fits a Ridge Regression model with five predictors:
- volatile acidity
- residual sugar
- chlorides
- free sulfur dioxide
- sulphates

Since we want to use CV to select the tuning parameter, we will use the `RidgeCV` function.

In [23]:
ridge_reg_cols = ['volatile acidity', 'residual sugar', 'chlorides',
                  'free sulfur dioxide', 'sulphates']
ridge_reg = RidgeCV(cv = 5, scoring = 'neg_mean_squared_error') \
            .fit(X_train1[ridge_reg_cols],
                 y_train1)

Now, we see which $\alpha$ is the best for this model (i.e., which $\alpha$ corresponds to the lowest MSE value). For completeness, we also print out the corresponding MSE value.

In [24]:
print("Best alpha:", ridge_reg.alpha_)
print("Corresponding MSE:", -ridge_reg.best_score_)

Best alpha: 10.0
Corresponding MSE: 1.0716378302726248


Therefore, the tuning parameter that should be selected for this Ridge Regression model is $\alpha = 10$. We now store the fit of this Ridge Regression model with the best tuning parameter for later use with the test data.

In [25]:
ridge_reg_best = Ridge(ridge_reg.alpha_, random_state = 44) \
             .fit(X_train1[ridge_reg_cols],
                  y_train1)

print(ridge_reg_best.intercept_)
print(np.array(list(zip(X_train1[ridge_reg_cols],
                        ridge_reg_best.coef_))))

10.467236867423512
[['volatile acidity' '-0.0404344356486099']
 ['residual sugar' '-0.4281520440468844']
 ['chlorides' '-0.3844788927380629']
 ['free sulfur dioxide' '-0.11990177144557149']
 ['sulphates' '0.0678072004851748']]


#### **Elastic Net**
The code below fits an Elastic Net model with five predictors:
- fixed acidity
- citric acid
- residual sugar
- total sulfur dioxide
- type_num

Since we want to use CV to select the tuning parameters, we will use the `ElasticNetCV` function.

In [26]:
en_reg_cols = ['fixed acidity', 'citric acid', 'residual sugar',
               'total sulfur dioxide', 'type_num']
en_reg = ElasticNetCV(cv = 5,
                      random_state = 44,
                      l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
                      n_alphas = 50) \
         .fit(X_train1[en_reg_cols],
              y_train1)

Now, we see which $\alpha$ and L1 ratio are best for this model.

In [27]:
print("Best alpha:", en_reg.alpha_)
print("Best L1 ratio:", en_reg.l1_ratio_)

Best alpha: 0.0008665468141444861
Best L1 ratio: 0.5


Therefore, the tuning parameter and L1 ratio that should be selected for this Elastic Net model are $\alpha \approx 0.0008665$ and L1 ratio = 0.5. We now store the fit of this Elastic Net model with the best tuning parameters, $\alpha$ and the L1 ratio, for later use with the test data.

In [28]:
en_reg_best = ElasticNet(alpha = en_reg.alpha_, l1_ratio = en_reg.l1_ratio_, random_state = 44) \
          .fit(X_train1[en_reg_cols],
               y_train1)
print(en_reg_best.intercept_)
print(np.array(list(zip(X_train1[en_reg_cols],
                        en_reg_best.coef_))))

10.467236867423512
[['fixed acidity' '-0.1335836744313908']
 ['citric acid' '0.10393897159224562']
 ['residual sugar' '-0.3635485951062797']
 ['total sulfur dioxide' '-0.4937674199557971']
 ['type_num' '-0.4173421115591611']]


### **Test Models**
In this sub-section, using the four selected models from the *Train Models* sub-section (`MLR_best`, `lasso_best`, `ridge_best`, `en_best`), we compare their performance on the test data. First, we need to standardize the test set using the same values we did with the training data.

*Note: Since we are overwriting our* `X_test1` *values with a function, this code block below can only be run once! Re-running will provide inaccurate results!*

In [29]:
#quick function to standardize based off of a supplied mean and standard deviation
def my_std_fun(x, means_reg, stds_reg):
    return(x-means_reg)/stds_reg

#loop through the test columns and use the function on each
for x in X_test1.columns:
    X_test1[x] = my_std_fun(X_test1[x], means_reg[x], stds_reg[x])
X_test1.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type_num
289,-0.008073,-0.671226,0.486276,1.837734,0.049741,1.047098,1.347853,1.078623,-1.478053,-0.346940,1.366087,-0.571351
4287,0.301865,-0.366794,-1.022364,-0.820422,-0.284639,-0.147994,0.015348,-0.631066,-0.857636,-0.681980,-0.940086,-0.571351
3391,-0.240527,-0.366794,-0.130895,0.593042,-0.841939,0.364188,-0.055719,-1.138524,-1.416012,-1.017020,2.519174,-0.571351
1451,0.456835,-0.123249,0.829149,-0.588361,1.331531,-1.286178,-1.761325,0.881090,0.693406,1.663297,1.366087,1.750237
3767,-0.240527,0.363842,-1.022364,1.563479,-0.646884,1.616189,1.241253,0.956016,-0.857636,-0.279932,-0.940086,-0.571351


Now that our test data has been standardized, we can compare the model performances.

In [30]:
MLR_reg_pred = MLR_best.predict(X_test1[MLR_best_cols])
lasso_reg_pred = lasso_reg_best.predict(X_test1[lasso_reg_cols])
ridge_reg_pred = ridge_reg_best.predict(X_test1[ridge_reg_cols])
en_reg_pred = en_reg_best.predict(X_test1[en_reg_cols])

#### **RMSE Model Metric**
The code prints out the RMSE on the test data for each model.

In [31]:
print('Test data RMSE for best MLR model is',
      np.sqrt(mean_squared_error(y_test1, MLR_reg_pred)))
print('Test data RMSE for best LASSO regression model is',
      np.sqrt(mean_squared_error(y_test1, lasso_reg_pred)))
print('Test data RMSE for best Ridge regression model is',
      np.sqrt(mean_squared_error(y_test1, ridge_reg_pred)))
print('Test data RMSE for best Elastic Net regression model is',
      np.sqrt(mean_squared_error(y_test1, en_reg_pred)))

Test data RMSE for best MLR model is 0.8137540182977578
Test data RMSE for best LASSO regression model is 0.818244239635236
Test data RMSE for best Ridge regression model is 1.0930273181065644
Test data RMSE for best Elastic Net regression model is 1.0797050184292556


Therefore, since RMSE is lowest for the MLR model, this is the best model choice when considering this specific model metric. As a reminder, this MLR model consists of the following predictors:
- `volatile acidity`
- `residual sugar`
- `total sulfur dioxide`
- `density`
- `type_num`

#### **MAE Model Metric**
The code prints out the MAE on the test data for each model.

In [32]:
print('Test data MAE for best MLR model is',
      mean_absolute_error(y_test1, MLR_reg_pred))
print('Test data MAE for best LASSO regression model is',
      mean_absolute_error(y_test1, lasso_reg_pred))
print('Test data MAE for best Ridge regression model is',
      mean_absolute_error(y_test1, ridge_reg_pred))
print('Test data MAE for best Elastic Net regression model is',
      mean_absolute_error(y_test1, en_reg_pred))

Test data MAE for best MLR model is 0.5515326259222215
Test data MAE for best LASSO regression model is 0.5718336328729148
Test data MAE for best Ridge regression model is 0.880354592415967
Test data MAE for best Elastic Net regression model is 0.8621391260498226


In concurrence with what was found with the RMSE, we see that the MAE is lowest for the MLR model. Thus, this is the best model choice when considering this specific model metric. As a reminder, this MLR model consists of the following predictors:
- `volatile acidity`
- `residual sugar`
- `total sulfur dioxide`
- `density`
- `type_num`

## **Classification Task: Wine Type as Response**
In this section, we use a variety of logistic regression techniques to fit both training models and tests models. We will use:
- Multiple logistic regression
- LASSO logistic regression
- Ridge logistic regression
- Elastic Net logistic regression

### **Train Models**
In this sub-section, we fit the four types of models described above on our training data.

#### **Multiple Logistic Regression**
Here are the four different multiple logistic regression models that I will choose to fit (listed predictors):
1. `citric acid`, `residual sugar`, `pH`
2. `volatile acidity`, `residual sugar`, `total sulfur dioxide`, `density`, `alcohol`
3. `fixed acidity`, `pH`, and their interaction
4. `residual sugar`, `density`, `residual sugar`$^{2}$, `density`$^{2}$, the interaction of density and residual sugar

##### **Multiple Logistic Regression #1**
The code below fits the first model described at the beginning of the Multiple Logistic Regression section.

In [33]:
MLR_1_log_cols = ['citric acid', 'residual sugar', 'pH']
MLR_1_log = linear_model.LogisticRegression(penalty = None, random_state = 44) \
            .fit(X_train2[MLR_1_log_cols], y_train2)
print(MLR_1_log.intercept_, MLR_1_log.coef_)

[-1.77401219] [[-0.13912618 -1.49811836  0.63612767]]


The order of the coefficients corresponds to the order of the columns in the `wine` dataframe. Therefore, the correct order is `citric acid`, `residual sugar`, and `pH`, respectively.

##### **Multiple Logistic Regression #2**
The code below fits the second model described at the beginning of the Multiple Logistic Regression section.

In [34]:
MLR_2_log_cols = ['volatile acidity', 'residual sugar', 'total sulfur dioxide',
                  'density', 'alcohol']
MLR_2_log = linear_model.LogisticRegression(penalty = None, random_state = 44) \
        .fit(X_train2[MLR_2_log_cols], y_train2)
print(MLR_2_log.intercept_, MLR_2_log.coef_)

[-4.08705994] [[ 1.37284218 -3.93222977 -2.39678466  6.07033747  2.6341746 ]]


The order of the coefficients corresponds to the order of the columns in the `wine` dataframe. Therefore, the correct order is `volatile acidity`, `residual sugar`, `total sulfur dioxide`, `density`, and `alcohol`, respectively.

##### **Multiple Logistic Regression #3**
The code below fits the third model described at the beginning of the Multiple Linear Regression section. This requires multiple steps since we are using an interaction term in this model.

In [35]:
MLR_3_log_cols = ['fixed acidity', 'pH']

#creates interaction object
poly3_log = PolynomialFeatures(interaction_only=True, include_bias = False)

#fits interaction object with variables we want
design3_log = poly3_log.fit_transform(X_train2[MLR_3_log_cols])

#fits MLR model with interactions
MLR_3_log = linear_model.LogisticRegression(penalty = None, random_state = 44) \
        .fit(X = design3_log, y = y_train2)

print(MLR_3_log.intercept_, MLR_3_log.coef_)

[-2.01016631] [[2.53734614 2.042283   0.08453471]]


We can use the `.get_feature_names_out()` method to confirm the order of the coefficients since we are using `PolynomialFeatures`.

In [36]:
poly3_log.get_feature_names_out(['fixed acidity', 'pH'])

array(['fixed acidity', 'pH', 'fixed acidity pH'], dtype=object)

Therefore, the order of the coefficients is `fixed acidity`, `pH`, and their interaction, respectively.

##### **Multiple Logistic Regression #4**
The code below fits the fourth and final model described at the beginning of the Multiple Logistic Regression section. This also requires multiple steps since we are using both quadratic terms and a interaction term in this model.

In [37]:
MLR_4_log_cols = ['residual sugar', 'density']

#creates interaction & polynomial object
poly4_log = PolynomialFeatures(degree = 2, interaction_only = False, include_bias = False)

#fits interaction object with variables we want
design4_log = poly4_log.fit_transform(X_train1[MLR_4_log_cols])

#fits MLR model with interactions
MLR_4_log = linear_model.LogisticRegression(penalty = None, random_state = 44) \
        .fit(X = design4_log, y = y_train2)

print(MLR_4_log.intercept_, MLR_4_log.coef_)

[-2.99714698] [[-4.04703701  3.75587792  0.1877809  -2.32773739  1.55727983]]


Again, we can use the `.get_feature_names_out()` method to confirm the order of the coefficients since we are using `PolynomialFeatures`.

In [38]:
poly4_log.get_feature_names_out(['residual sugar', 'density'])

array(['residual sugar', 'density', 'residual sugar^2',
       'residual sugar density', 'density^2'], dtype=object)

Therefore, the order of the coefficients is `residual sugar`, `density`, `residual sugar`$^{2}$, the interaction between residual sugar and density, and `density`$^{2}$, respectively.

##### **CV & Negative Log Loss with Multiple Logistic Regression Models**
The code below uses CV & negative log loss with each of the above multiple logistic regression models to help select the best one.

In [39]:
#cross-validation on multiple logistic regression #1
cv1_log = cross_validate(linear_model.LogisticRegression(penalty = None, random_state = 44),
                         X_train2[MLR_1_log_cols],
                         y_train2,
                         cv = 5,
                         scoring = 'neg_log_loss',
                         return_train_score = True)

#cross-validation on multiple logistic regression #2
cv2_log = cross_validate(linear_model.LogisticRegression(penalty = None, random_state = 44),
                         X_train2[MLR_2_log_cols],
                         y_train2,
                         cv = 5,
                         scoring = 'neg_log_loss',
                         return_train_score = True)

#cross-validation on multiple logistic regression #3
cv3_log = cross_validate(linear_model.LogisticRegression(penalty = None, random_state = 44),
                         design3_log,
                         y_train2,
                         cv = 5,
                         scoring = 'neg_log_loss',
                         return_train_score = True)

#cross-validation on multiple logistic regression #4
cv4_log = cross_validate(linear_model.LogisticRegression(penalty = None, random_state = 44),
                         design4_log,
                         y_train2,
                         cv = 5,
                         scoring = 'neg_log_loss',
                         return_train_score = True)

Now that cross-validation is complete, we can compare the CV negative log-loss values to select the best model. Since I chose to use *negative* log-loss, closer to 0 is better!

In [40]:
print(
    round(cv1_log['test_score'].mean(),4),
    round(cv2_log['test_score'].mean(),4),
    round(cv3_log['test_score'].mean(),4),
    round(cv4_log['test_score'].mean(),4)
    )

-0.434 -0.0359 -0.28 -0.146


Since the negative log-loss is closest to 0 for the second multiple logistic regression model, that is the best choice of the four fitted models!

In [41]:
log_best_cols = MLR_2_log_cols
log_best = MLR_2_log
print(log_best.intercept_)
print(np.array(list(zip(log_best_cols, log_best.coef_[0]))))

[-4.08705994]
[['volatile acidity' '1.3728421836631224']
 ['residual sugar' '-3.932229770693349']
 ['total sulfur dioxide' '-2.3967846550971474']
 ['density' '6.070337473526581']
 ['alcohol' '2.634174597624738']]


#### **LASSO Logistic Regression**
The code below fits a LASSO logistic regression model with five predictors:
- fixed acidity
- citric acid
- density
- pH
- quality

We will use `LogisticRegressionCV` and specify `penalty = l1` to ensure a LASSO model is fit.

In [44]:
lasso_log_cols = ['fixed acidity', 'citric acid', 'density', 'pH', 'quality']
lasso_log = LogisticRegressionCV(cv = 5,
                                 solver = 'saga',
                                 penalty = 'l1', #L1 corresponds to LASSO
                                 Cs = 250,
                                 scoring = 'neg_log_loss',
                                 random_state = 44) \
            .fit(X_train2[lasso_log_cols],
                 y_train2)

Now, we find the optimal regularization value (smaller = more regularized!).

In [43]:
lasso_log.C_

array([0.25446174])

Therefore, the optimal regularization value that should be selected for this LASSO logistic regression model is $C = 0.25446174$. We now store the fit of this LASSO logistc regression model with the best $C$ value for later use with the test data.

In [46]:
lasso_log_best = LogisticRegression(solver = 'saga',
                                    penalty = 'l1', #L1 corresponds to LASSO
                                    C = lasso_log.C_[0], #optimal regularization value
                                    random_state = 44
                                    ).fit(X_train2[lasso_log_cols], y_train2)

print(lasso_log_best.intercept_)
print(np.array(list(zip(lasso_log_cols, lasso_log_best.coef_[0]))))

[-2.39493559]
[['fixed acidity' '2.814443969437007']
 ['citric acid' '-1.3242757753837897']
 ['density' '0.5970434968253326']
 ['pH' '1.7788229332076333']
 ['quality' '-0.0039567536564411145']]


#### **Ridge Logistic Regression**
The code below fits a Ridge Regression model with five predictors:
- volatile acidity
- residual sugar
- chlorides
- free sulfur dioxide
- sulphates

We will use `LogisticRegressionCV` and specify `penalty = l2` to ensure a Ridge model is fit.

In [47]:
ridge_log_cols = ['volatile acidity', 'residual sugar', 'chlorides',
                  'free sulfur dioxide', 'sulphates']
ridge_log = LogisticRegressionCV(cv = 5,
                                 solver = 'saga',
                                 penalty = 'l2', #L2 corresponds to Ridge
                                 Cs = 250,
                                 scoring = 'neg_log_loss',
                                 random_state = 44) \
            .fit(X_train2[ridge_log_cols],
                 y_train2)

Now, we find the optimal regularization value.

In [48]:
ridge_log.C_

array([0.96368643])

Therefore, the optimal regularization value that should be selected for this Ridge logistic regression model is $C = 0.96368643$. We now store the fit of this Ridge logistc regression model with the best $C$ value for later use with the test data.

In [49]:
ridge_log_best = LogisticRegression(solver = 'saga',
                                    penalty = 'l2', #L2 corresponds to Ridge
                                    C = ridge_log.C_[0], #optimal regularization value
                                    random_state = 44
                                    ).fit(X_train2[ridge_log_cols], y_train2)

print(ridge_log_best.intercept_)
print(np.array(list(zip(ridge_log_cols, ridge_log_best.coef_[0]))))

[-3.81139316]
[['volatile acidity' '1.9702295715420748']
 ['residual sugar' '-1.8783637857353186']
 ['chlorides' '1.485436648878926']
 ['free sulfur dioxide' '-1.3769230913807726']
 ['sulphates' '1.739774185477613']]


#### **Elastic Net Logistic Regression**
The code below fits an Elastic Net model with five predictors:
- fixed acidity
- citric acid
- residual sugar
- total sulfur dioxide
- alcohol

We will use `LogisticRegressionCV` and specify `penalty = elasticnet` to ensure an Elastic Net model is fit. To speed up the runtime, I will use `Cs = 100` and a reduced list of `l1_ratios` compared to what has been seen in earlier parts of the assignment.

In [58]:
en_log_cols = ['fixed acidity', 'citric acid', 'residual sugar',
               'total sulfur dioxide', 'alcohol']
en_log = LogisticRegressionCV(cv = 5,
                              solver = 'saga',
                              penalty = 'elasticnet', #elasticnet corresponds to ElasticNet
                              Cs = 100,
                              scoring = 'neg_log_loss',
                              random_state = 44,
                              l1_ratios = [0.1, 0.25, 0.5, 0.75, 0.9, 1]) \
         .fit(X_train2[en_log_cols], y_train2)

Now, we see which $C$ and L1 ratio are best for this model.

In [59]:
print("Best alpha:", en_log.C_)
print("Best L1 ratio:", en_log.l1_ratio_)

Best alpha: [0.91116276]
Best L1 ratio: [0.1]


Therefore, the optimal regularization value that should be selected for this Elastic Net logistic regression model is $C = 0.91116276$, and the optimal L1 ratio is 0.1. We now store the fit of this Elastic Net logistc regression model with these parameter values for later use with the test data.

In [60]:
en_log_best = LogisticRegression(solver = 'saga',
                                    penalty = 'elasticnet', #elasticnet corresponds to Elastic Net
                                    C = en_log.C_[0], #optimal regularization value
                                    random_state = 44,
                                 l1_ratio = 0.1
                                    ).fit(X_train2[en_log_cols], y_train2)

print(en_log_best.intercept_)
print(np.array(list(zip(en_log_cols, en_log_best.coef_[0]))))

[-3.00053206]
[['fixed acidity' '1.418744941617832']
 ['citric acid' '-0.8337938323489078']
 ['residual sugar' '-0.7521512857390797']
 ['total sulfur dioxide' '-2.963251449704208']
 ['alcohol' '-0.6258116301077278']]


### **Test Models**
In this sub-section, using the four selected models from the *Train Models* sub-section (`log_best`, `lasso_log_best`, `ridge_log_best`, `en_lolg_best`), we compare their performance on the test data. First, we need to standardize the test set using the same values we did with the training data.

*Note: Since we are overwriting our* `X_test2` *values with a function, this code block below can only be run once! Re-running will provide inaccurate results!*

In [61]:
#quick function to standardize based off of a supplied mean and standard deviation
def my_std_fun(x, means_cls, stds_cls):
    return(x-means_cls)/stds_cls

#loop through the test columns and use the function on each
for x in X_test2.columns:
    X_test2[x] = my_std_fun(X_test2[x], means_cls[x], stds_cls[x])
X_test2.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
289,-0.008073,-0.671226,0.486276,1.837734,0.049741,1.047098,1.347853,1.078623,-1.478053,-0.346940,-1.242573,1.366087
4287,0.301865,-0.366794,-1.022364,-0.820422,-0.284639,-0.147994,0.015348,-0.631066,-0.857636,-0.681980,-0.395693,-0.940086
3391,-0.240527,-0.366794,-0.130895,0.593042,-0.841939,0.364188,-0.055719,-1.138524,-1.416012,-1.017020,1.552130,2.519174
1451,0.456835,-0.123249,0.829149,-0.588361,1.331531,-1.286178,-1.761325,0.881090,0.693406,1.663297,0.451186,1.366087
3767,-0.240527,0.363842,-1.022364,1.563479,-0.646884,1.616189,1.241253,0.956016,-0.857636,-0.279932,-0.903821,-0.940086


Now that our test data has been standardized, we can compare the model performances.

#### **Log-Loss Model Metric**
The code below computes the prediction models on the test data for use with the log-loss model metric.

In [77]:
log_pred_LL = log_best.predict_proba(X_test2[log_best_cols])
lasso_log_pred_LL = lasso_log_best.predict_proba(X_test2[lasso_log_cols])
ridge_log_pred_LL = ridge_log_best.predict_proba(X_test2[ridge_log_cols])
en_log_pred_LL = en_log_best.predict_proba(X_test2[en_log_cols])

Now, we print out the log-loss for each model. *Lower* is better!

In [81]:
print('Test data log-loss for best multiple logistic regression model is',
      log_loss(y_test2, log_pred_LL))
print('Test data log-loss for best LASSO logistic regression model is',
      log_loss(y_test2, lasso_log_pred_LL))
print('Test data log-loss for best Ridge logistic regression model is',
      log_loss(y_test2, ridge_log_pred_LL))
print('Test data log-loss for best Elastic Net logistic regression model is',
      log_loss(y_test2, en_log_pred_LL))

Test data log-loss for best multiple logistic regression model is 0.06212040947751372
Test data log-loss for best LASSO logistic regression model is 0.23699048440496084
Test data log-loss for best Ridge logistic regression model is 0.13676742437295208
Test data log-loss for best Elastic Net logistic regression model is 0.1302858586363949


Therefore, since the log-loss is lowest for the multiple logistic regression model, this is the best model choice when considering this specific model metric. As a reminder, this multiple logistic regression model consists of the following predictors:
- `volatile acidity`
- `residual sugar`
- `total sulfur dioxide`
- `density`
- `alcohol`

#### **Accuracy Score Model Metric**
The code below computes the prediction models on the test data for use with the accuracy score model metric.

In [83]:
log_pred_AS = log_best.predict(X_test2[log_best_cols])
lasso_log_pred_AS = lasso_log_best.predict(X_test2[lasso_log_cols])
ridge_log_pred_AS = ridge_log_best.predict(X_test2[ridge_log_cols])
en_log_pred_AS = en_log_best.predict(X_test2[en_log_cols])

Now, we print out the acuracy score for each model. *Higher* is better!

In [84]:
print('Test data accuracy score for best multiple logistic regression model is',
      accuracy_score(y_test2, log_pred_AS))
print('Test data log-loss for best LASSO logistic regression model is',
      accuracy_score(y_test2, lasso_log_pred_AS))
print('Test data log-loss for best Ridge logistic regression model is',
      accuracy_score(y_test2, ridge_log_pred_AS))
print('Test data log-loss for best Elastic Net logistic regression model is',
      accuracy_score(y_test2, en_log_pred_AS))

Test data accuracy score for best multiple logistic regression model is 0.9930769230769231
Test data log-loss for best LASSO logistic regression model is 0.8984615384615384
Test data log-loss for best Ridge logistic regression model is 0.9538461538461539
Test data log-loss for best Elastic Net logistic regression model is 0.9515384615384616


In concurrence with what was found with the log-loss, we see that the accuracy score is highest for the multiple logistic regression model. Thus, this is the best model choice when considering this specific model metric. As a reminder, this multiple logistic regression model consists of the following predictors:
- `volatile acidity`
- `residual sugar`
- `total sulfur dioxide`
- `density`
- `alcohol`